# Hyperparameter Tuning with MLflow

## Topic Goal
Track multiple parameter combinations.


## 1. Import Libraries

In [ ]:
# Import os to create folders and clean MLflow project environment variables.
import os

# Import warnings to keep notebook output clean.
import warnings

# Suppress non-critical warnings.
warnings.filterwarnings("ignore")

# Import MLflow for tracking, model logging, and model registry.
import mlflow

# Import sklearn flavor support for logging scikit-learn pipelines.
import mlflow.sklearn

# Import infer_signature to capture model input and output schema.
from mlflow.models.signature import infer_signature

# Import pandas for CSV data processing.
import pandas as pd

# Import NumPy for numerical operations.
import numpy as np

# Import matplotlib for artifact charts.
import matplotlib.pyplot as plt

# Import hashlib for dataset versioning examples.
import hashlib

# Import requests for REST client examples.
import requests

# Import train-test split utility.
from sklearn.model_selection import train_test_split

# Import preprocessing utilities.
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Import Pipeline to package preprocessing and model together.
from sklearn.pipeline import Pipeline

# Import model algorithms.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier

# Import classification metrics.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report


## 2. Configure MLflow

In [ ]:
EXPERIMENT_NAME = "advanced_03_hyperparameter_tuning"
RUN_NAME = "03_hyperparameter_tuning_same_run_usecase"

# Remove inherited MLflow Project variables to avoid experiment ID issues.
os.environ.pop("MLFLOW_EXPERIMENT_ID", None)
os.environ.pop("MLFLOW_RUN_ID", None)

# Create local folders.
os.makedirs("mlruns", exist_ok=True)
os.makedirs("artifacts", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

# Configure local tracking store.
mlflow.set_tracking_uri("file:./mlruns")

# Define dataset paths.
DATA_PATH = "datasets/customer_churn_500.csv"
CURRENT_DATA_PATH = "datasets/customer_churn_current_500.csv"

# Set experiment.
mlflow.set_experiment(EXPERIMENT_NAME)

# Create registered model name for this topic.
REGISTERED_MODEL_NAME = EXPERIMENT_NAME.replace("-", "_").replace(" ", "_") + "_Model"

# Print setup details.
print("Experiment:", EXPERIMENT_NAME)
print("Registered Model Name:", REGISTERED_MODEL_NAME)
print("Tracking URI:", mlflow.get_tracking_uri())


## 3. Load and Prepare Dataset

In [ ]:
# Load baseline dataset.
df = pd.read_csv(DATA_PATH)

# Display first five records.
display(df.head())

# Define target column.
target_column = "churn"

# Create feature matrix.
X = df.drop(columns=["customer_id", target_column])

# Create target vector.
y = df[target_column]

# Identify categorical columns.
categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()

# Identify numerical columns.
numerical_columns = X.select_dtypes(exclude=["object"]).columns.tolist()

# Create preprocessing transformer.
preprocessor = ColumnTransformer(
    transformers=[
        # Scale numerical features.
        ("num", StandardScaler(), numerical_columns),

        # One-hot encode categorical features.
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
    ]
)

# Split data.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Print data summary.
print("Shape:", df.shape)
print("Categorical:", categorical_columns)
print("Numerical:", numerical_columns)


## 4. Topic-Specific Setup

In [ ]:
# Define the hyperparameter search space.
param_grid = [
    # Trial 1 uses fewer trees and a shallow tree depth.
    {"n_estimators": 100, "max_depth": 4},

    # Trial 2 uses medium trees and medium depth.
    {"n_estimators": 150, "max_depth": 6},

    # Trial 3 uses more trees and deeper trees.
    {"n_estimators": 200, "max_depth": 8},

    # Trial 4 uses the highest tree count and deepest setting in this demo.
    {"n_estimators": 250, "max_depth": 10},
]

# Create an empty list to store tuning trial results.
tuning_results = []

# Create a topic artifact explaining what this notebook does.
with open("outputs/topic_artifact.txt", "w") as file:
    # Write the tuning purpose.
    file.write("Hyperparameter tuning tries multiple parameter combinations and compares metrics.\n")

    # Write the parameters being tuned.
    file.write("Parameters tuned: n_estimators and max_depth.\n")

# Display the search space.
pd.DataFrame(param_grid)


## 5. Hyperparameter Tuning Trials

This section performs actual hyperparameter tuning.  
Each trial is logged as a separate MLflow run so students can compare parameter combinations in MLflow UI.

In [ ]:
# Loop through every hyperparameter combination.
for params in param_grid:

    # Create a Random Forest model using the current parameter combination.
    trial_model = RandomForestClassifier(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        random_state=42
    )

    # Create a complete pipeline for this trial.
    trial_pipeline = Pipeline(steps=[
        # Apply preprocessing to categorical and numerical columns.
        ("preprocessor", preprocessor),

        # Train the Random Forest model with current parameters.
        ("model", trial_model),
    ])

    # Start one MLflow run for this tuning trial.
    with mlflow.start_run(run_name=f"trial_trees_{params['n_estimators']}_depth_{params['max_depth']}"):

        # Log number of trees.
        mlflow.log_param("n_estimators", params["n_estimators"])

        # Log maximum tree depth.
        mlflow.log_param("max_depth", params["max_depth"])

        # Log dataset path.
        mlflow.log_param("dataset", DATA_PATH)

        # Train the trial pipeline.
        trial_pipeline.fit(X_train, y_train)

        # Generate trial predictions.
        trial_predictions = trial_pipeline.predict(X_test)

        # Calculate trial accuracy.
        trial_accuracy = accuracy_score(y_test, trial_predictions)

        # Calculate trial precision.
        trial_precision = precision_score(y_test, trial_predictions, zero_division=0)

        # Calculate trial recall.
        trial_recall = recall_score(y_test, trial_predictions, zero_division=0)

        # Calculate trial F1-score.
        trial_f1 = f1_score(y_test, trial_predictions, zero_division=0)

        # Log trial accuracy.
        mlflow.log_metric("accuracy", trial_accuracy)

        # Log trial precision.
        mlflow.log_metric("precision", trial_precision)

        # Log trial recall.
        mlflow.log_metric("recall", trial_recall)

        # Log trial F1-score.
        mlflow.log_metric("f1_score", trial_f1)

        # Store trial result locally.
        tuning_results.append({
            "n_estimators": params["n_estimators"],
            "max_depth": params["max_depth"],
            "accuracy": trial_accuracy,
            "precision": trial_precision,
            "recall": trial_recall,
            "f1_score": trial_f1,
        })

# Convert tuning results into a dataframe.
tuning_results_df = pd.DataFrame(tuning_results)

# Sort results by F1-score in descending order.
tuning_results_df = tuning_results_df.sort_values("f1_score", ascending=False)

# Save tuning results as CSV artifact for the final run.
tuning_results_df.to_csv("outputs/hyperparameter_tuning_results.csv", index=False)

# Display tuning results.
display(tuning_results_df)

# Select the best parameter row.
best_row = tuning_results_df.iloc[0]

# Convert best parameters to integers.
best_n_estimators = int(best_row["n_estimators"])
best_max_depth = int(best_row["max_depth"])

# Print the best parameter combination.
print("Best n_estimators:", best_n_estimators)
print("Best max_depth:", best_max_depth)
print("Best F1-score:", best_row["f1_score"])


## 6. Train Best Model, Infer Signature, Log Model, and Register from the SAME Run

This section trains the final model using the best hyperparameters.  
The final model, signature, input example, metrics, artifacts, model URI, and registry registration are connected to the same MLflow run.

In [ ]:
# Create the best Random Forest model using the selected tuning parameters.
model = RandomForestClassifier(
    n_estimators=best_n_estimators,
    max_depth=best_max_depth,
    random_state=42
)

# Create the final pipeline using the best model.
pipeline = Pipeline(steps=[
    # Apply preprocessing to raw input columns.
    ("preprocessor", preprocessor),

    # Train the best model.
    ("model", model),
])

# Train the final pipeline.
pipeline.fit(X_train, y_train)

# Generate final predictions.
predictions = pipeline.predict(X_test)

# Calculate final accuracy.
accuracy = accuracy_score(y_test, predictions)

# Calculate final precision.
precision = precision_score(y_test, predictions, zero_division=0)

# Calculate final recall.
recall = recall_score(y_test, predictions, zero_division=0)

# Calculate final F1-score.
f1 = f1_score(y_test, predictions, zero_division=0)

# Create input example for model signature.
input_example = X_test.head(5)

# Generate sample predictions for signature.
sample_predictions = pipeline.predict(input_example)

# Infer model input and output signature.
signature = infer_signature(input_example, sample_predictions)

# Start the SAME MLflow run for best params, metrics, artifacts, signature, model, and model URI.
with mlflow.start_run(run_name=RUN_NAME):

    # Log dataset path.
    mlflow.log_param("dataset", DATA_PATH)

    # Log topic.
    mlflow.log_param("topic", EXPERIMENT_NAME)

    # Log final selected number of trees.
    mlflow.log_param("best_n_estimators", best_n_estimators)

    # Log final selected max depth.
    mlflow.log_param("best_max_depth", best_max_depth)

    # Log model family.
    mlflow.log_param("model_family", "RandomForestClassifier")

    # Log final accuracy.
    mlflow.log_metric("accuracy", accuracy)

    # Log final precision.
    mlflow.log_metric("precision", precision)

    # Log final recall.
    mlflow.log_metric("recall", recall)

    # Log final F1-score.
    mlflow.log_metric("f1_score", f1)

    # Log tuning results CSV as artifact.
    mlflow.log_artifact("outputs/hyperparameter_tuning_results.csv")

    # Log topic explanation artifact if available.
    if os.path.exists("outputs/topic_artifact.txt"):
        mlflow.log_artifact("outputs/topic_artifact.txt")

    # Log final best model WITH signature and input example in the SAME run.
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="model",
        input_example=input_example,
        signature=signature
    )

    # Create model URI from the same active run.
    model_uri = f"runs:/{mlflow.active_run().info.run_id}/model"

# Register the model using the SAME run's model URI.
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=REGISTERED_MODEL_NAME
)

# Print final model URI.
print("Same-run model URI:", model_uri)

# Print registered model name.
print("Registered model:", registered_model.name)

# Print registered model version.
print("Registered version:", registered_model.version)

# Print final F1-score.
print("Final best model F1-score:", f1)




1. Multiple tuning trial runs, one for each hyperparameter combination.
2. One final run named like `*_same_run_usecase`.
3. The final run contains the best parameters, final metrics, tuning results artifact, model signature, input example, model artifact, and registry source.

This keeps the tuning process visible while keeping signature and registry tied to the same final model run.